# Global Context Architecture Sweep

This notebook tests whether the winning autoencoder-pretrained contrastive recipe improves with ResNet, multi-scale, dilation, or global-context encoder variants. Each architecture gets a matching autoencoder pretraining run before contrastive training, so incompatible checkpoint loading does not contaminate the comparison.

Use this notebook for architecture exploration. It is broader than the final best-recipe run and intentionally sweeps alternatives.


In [ ]:
import os, sys, time, json, platform, subprocess, shutil
from pathlib import Path
print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)


In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nilearn', 'nibabel', 'huggingface-hub', 'safetensors', 'adapters', 'transformers', 'pyarrow', 'matplotlib', 'pandas', 'scikit-learn', 'tqdm', 'umap-learn'])
REPO_URL = os.environ.get('NEUROVLM_REPO_URL', 'https://github.com/neurovlm/neurovlm.git')
REPO_BRANCH = os.environ.get('NEUROVLM_REPO_BRANCH', 'neurovlm_gnn')
REPO_DIR = os.environ.get('NEUROVLM_REPO_DIR', '/content/neurovlm_gnn')
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'checkout', REPO_BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH])
os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[viz,notebook,metrics]'])
sys.path.insert(0, str(Path(REPO_DIR) / 'experiments' / '3dcnn'))
sys.path.insert(0, str(Path(REPO_DIR) / 'src'))
sys.path.insert(0, REPO_DIR)
from atlas_free_cnn.notebook_utils import (
    discover_unified_split_dir as shared_discover_unified_split_dir,
    download_text_embedding_cache,
    required_text_records_from_jsonls,
    resolve_text_embedding_cache,
    text_embedding_metadata_fields,
    validate_text_embedding_cache,
)
print('Working directory:', os.getcwd())


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

DRIVE_ROOT = '/content/drive/MyDrive/neurovlm'
RUNS_DIR = f'{DRIVE_ROOT}/runs_ale_3dcnn_best_global'
CACHE_DIR = f'{DRIVE_ROOT}/data_ale_3dcnn/ale_caches'
LOCAL_CACHE_DIR = '/content/ale_caches_ale_3dcnn'
EVAL_RESOURCE_DIR = '/content/drive/MyDrive/neurovlm_evaluation_resources'
RUN_STAMP = time.strftime('%Y%m%d_%H%M%S')
HF_DATASET_REPO = os.environ.get('NEUROVLM_ATLAS_FREE_HF_REPO', 'neurovlm/atlas_free_cnn_dataset')
LOCAL_UNIFIED_CACHE_DIR = Path(REPO_DIR) / 'experiments' / '3dcnn' / 'atlas_free_cnn' / 'cache' / 'unified_jsonl_rebuild'
LOCAL_SPLIT_DIR = LOCAL_UNIFIED_CACHE_DIR / 'splits'
LOCAL_PACK_DIR = Path(REPO_DIR) / 'experiments' / '3dcnn' / 'atlas_free_cnn' / 'cache' / 'hf_atlas_free_cnn_rebuild'
LOCAL_TEXT_CACHE_DIR = Path(REPO_DIR) / 'experiments' / '3dcnn' / 'atlas_free_cnn' / 'cache' / 'text_embeddings'
TEXT_EMBEDDING_CONVENTION = os.environ.get('NEUROVLM_TEXT_EMBEDDING_CONVENTION', 'normalized_specter2')
USE_UNIFIED_JSONL_STAGE3 = os.environ.get('NEUROVLM_USE_UNIFIED_JSONL_STAGE3', '1') != '0'
TEXT_EMBEDDING_SPEC = resolve_text_embedding_cache(
    TEXT_EMBEDDING_CONVENTION,
    repo_dir=REPO_DIR,
    local_cache_dir=LOCAL_TEXT_CACHE_DIR,
    hf_repo=HF_DATASET_REPO,
)
TEXT_EMBEDDING_CACHE = Path(TEXT_EMBEDDING_SPEC['local_cache_path'])
TEXT_EMBEDDING_AUDIT = None
for d in [RUNS_DIR, CACHE_DIR, LOCAL_CACHE_DIR, EVAL_RESOURCE_DIR, LOCAL_TEXT_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)
if USE_UNIFIED_JSONL_STAGE3:
    UNIFIED_SPLIT_DIR = shared_discover_unified_split_dir(
        repo_dir=REPO_DIR,
        drive_root=DRIVE_ROOT,
        dataset_repo=HF_DATASET_REPO,
        local_unified_cache_dir=LOCAL_UNIFIED_CACHE_DIR,
        local_split_dir=LOCAL_SPLIT_DIR,
        local_pack_dir=LOCAL_PACK_DIR,
    )
    TRAIN_JSONL = UNIFIED_SPLIT_DIR / 'train.jsonl'
    VAL_JSONL = UNIFIED_SPLIT_DIR / 'val.jsonl'
    TEST_JSONL = UNIFIED_SPLIT_DIR / 'test.jsonl'
    if not TEXT_EMBEDDING_CACHE.exists():
        TEXT_EMBEDDING_CACHE = download_text_embedding_cache(TEXT_EMBEDDING_SPEC)
        TEXT_EMBEDDING_SPEC['local_cache_path'] = str(TEXT_EMBEDDING_CACHE)
    requirements = required_text_records_from_jsonls([TRAIN_JSONL, VAL_JSONL, TEST_JSONL])
    TEXT_EMBEDDING_AUDIT = validate_text_embedding_cache(
        TEXT_EMBEDDING_SPEC,
        required_text_ids=requirements['text_ids'],
        required_texts=requirements['texts'],
        expected_dim=TEXT_EMBEDDING_SPEC['expected_dim'],
        expect_unit_norm=TEXT_EMBEDDING_SPEC['expect_unit_norm'],
    )
else:
    UNIFIED_SPLIT_DIR = TRAIN_JSONL = VAL_JSONL = TEST_JSONL = None
TEXT_EMBEDDING_METADATA = text_embedding_metadata_fields(TEXT_EMBEDDING_SPEC, TEXT_EMBEDDING_AUDIT)
with (Path(RUNS_DIR) / f'text_embedding_config_{RUN_STAMP}.json').open('w') as f:
    json.dump({
        'use_unified_jsonl_stage3': USE_UNIFIED_JSONL_STAGE3,
        'train_jsonl': str(TRAIN_JSONL or ''),
        'val_jsonl': str(VAL_JSONL or ''),
        'test_jsonl': str(TEST_JSONL or ''),
        **TEXT_EMBEDDING_METADATA,
        'validation': (TEXT_EMBEDDING_AUDIT or {}).get('stats', {}),
    }, f, indent=2)

# Keep Drive evaluation resources in sync with the repo copy used by this notebook.
# This matters for network-term definitions, because the evaluator reads from EVAL_RESOURCE_DIR first.
repo_network_resource_dir = Path(REPO_DIR) / 'experiments' / 'evaluation_resources' / 'networks_labels'
drive_network_resource_dir = Path(EVAL_RESOURCE_DIR) / 'networks_labels'
drive_network_resource_dir.mkdir(parents=True, exist_ok=True)
for resource_name in [
    'network_test_set_labels.csv',
    'network_terms_with_definitions.csv',
    'network_terms_with_definitions.json',
]:
    src = repo_network_resource_dir / resource_name
    if src.exists():
        dst = drive_network_resource_dir / resource_name
        shutil.copy2(src, dst)
        print('Synced evaluation resource:', dst)

def find_autoencoder_checkpoint():
    for env_name in ['ALE_CNN_AE_CHECKPOINT', 'AUTOENCODER_CHECKPOINT']:
        env_path = os.environ.get(env_name, '')
        if env_path and Path(env_path).expanduser().exists():
            return str(Path(env_path).expanduser())
    # This original plain-CNN checkpoint is the one that previously loaded and ran cleanly.
    # Keep it as the Stage 1 default; newly trained ResNet/global-context AEs still use their best checkpoint.
    exact_last = Path(DRIVE_ROOT) / 'runs_ale_3dcnn_autoencoder_pretrain' / 'autoencoder_atlas_free_20260510_194641' / 'checkpoints' / 'last_cnn_autoencoder.pt'
    if exact_last.exists():
        return str(exact_last)
    exact_best = exact_last.with_name('best_cnn_autoencoder.pt')
    if exact_best.exists():
        return str(exact_best)
    candidates = []
    roots = [
        Path(DRIVE_ROOT) / 'runs_ale_3dcnn_autoencoder_pretrain',
        Path(DRIVE_ROOT) / 'runs_ale_3dcnn_full_pipeline',
        Path(DRIVE_ROOT) / 'runs_ale_3dcnn_best_global',
        Path('runs_ale_3dcnn_autoencoder_pretrain'),
    ]
    best_patterns = [
        'autoencoder_atlas_free_*/checkpoints/best_cnn_autoencoder.pt',
        'stage1_autoencoder_atlas_free_*/checkpoints/best_cnn_autoencoder.pt',
        '**/checkpoints/best_cnn_autoencoder.pt',
        '**/checkpoints/best_generation_top5_dice.pt',
    ]
    fallback_patterns = [
        'autoencoder_atlas_free_*/checkpoints/last_cnn_autoencoder.pt',
        'stage1_autoencoder_atlas_free_*/checkpoints/last_cnn_autoencoder.pt',
        '**/checkpoints/last_cnn_autoencoder.pt',
    ]
    for patterns in [best_patterns, fallback_patterns]:
        candidates = []
        for root in roots:
            if root.exists():
                for pattern in patterns:
                    candidates.extend(root.glob(pattern))
        existing = [p for p in candidates if p.exists()]
        if existing:
            return str(max(existing, key=lambda p: p.stat().st_mtime))
    return ''

AUTOENCODER_CHECKPOINT = find_autoencoder_checkpoint()
# Optional manual override in a notebook cell:
# AUTOENCODER_CHECKPOINT = '/content/drive/MyDrive/neurovlm/.../checkpoints/last_cnn_autoencoder.pt'
CACHE_FILE = f'{LOCAL_CACHE_DIR}/atlas_free_ale_4mm_fwhm9p0_crop_float16.pt'
COMPARISON_FILE = f'{RUNS_DIR}/best_global_context_comparison.csv'
CONTRASTIVE_BATCH_CANDIDATES = os.environ.get(
    'ALE_CNN_BATCH_CANDIDATES',
    '2048,1536,1024,768,512,384,256,192,128,96,64,32,16,8,4',
)
AE_BATCH_CANDIDATES = os.environ.get(
    'ALE_CNN_AE_BATCH_CANDIDATES',
    '256,192,128,96,64,32,16,8,4',
)

AE_EPOCHS = int(os.environ.get('ALE_CNN_AE_EPOCHS', '150'))
CONTRASTIVE_EPOCHS = int(os.environ.get('ALE_CNN_CONTRASTIVE_EPOCHS', '200'))
SKIP_COMPLETED = os.environ.get('ALE_CNN_SKIP_COMPLETED', '1') != '0'
print('AUTOENCODER_CHECKPOINT:', AUTOENCODER_CHECKPOINT or '<not found yet>')
if not AUTOENCODER_CHECKPOINT:
    print('Set ALE_CNN_AE_CHECKPOINT or AUTOENCODER_CHECKPOINT to your Drive checkpoint before run_variant().')


In [ ]:
def base_args(autoencoder_checkpoint, epochs=None):
    args = [
        '--mode', 'atlas_free',
        '--epochs', str(CONTRASTIVE_EPOCHS if epochs is None else epochs),
        '--batch-size-auto', '--batch-size-candidates', CONTRASTIVE_BATCH_CANDIDATES,
        '--lr-cnn', '1e-4', '--lr-proj', '1e-5',
        '--warmup-epochs', '5', '--temperature', '0.07',
        '--val-interval', '5', '--early-stopping-patience', '25',
        '--out-dim', '384', '--dropout', '0.1', '--norm', 'group',
        '--kernel-fwhm-mm', '9', '--resolution-mm', '4', '--cache-dtype', 'float16',
        '--cache-file', CACHE_FILE,
        '--encoder-init', 'autoencoder_pretrained', '--autoencoder-checkpoint', autoencoder_checkpoint,
        '--text-proj-init', 'pretrained_infonce',
        '--semantic-eval', '--eval-resource-dir', EVAL_RESOURCE_DIR,
        '--train-sanity-n', '512', '--num-workers', '0',
        '--comparison-file', COMPARISON_FILE,
    ]
    if USE_UNIFIED_JSONL_STAGE3:
        args.extend([
            '--train-jsonl', TRAIN_JSONL,
            '--val-jsonl', VAL_JSONL,
            '--test-jsonl', TEST_JSONL,
            '--text-embedding-cache', TEXT_EMBEDDING_CACHE,
            '--domain', 'pubmed',
            '--target-shape', '36,45,38',
        ])
    return args


def autoencoder_args(v, run_dir):
    args = [
        '--mode', 'atlas_free',
        '--model', v['model'],
        '--epochs', str(v.get('ae_epochs', AE_EPOCHS)),
        '--batch-size-auto', '--batch-size-candidates', v.get('ae_batch_size_candidates', AE_BATCH_CANDIDATES),
        '--lr', str(v.get('ae_lr', 3e-4)), '--weight-decay', '1e-4',
        '--val-interval', '5', '--early-stopping-patience', '20',
        '--base-channels', str(v['base_channels']),
        '--num-blocks', str(v['num_blocks']),
        '--blocks-per-stage', str(v.get('blocks_per_stage', 2)),
        '--latent-dim', '384', '--dropout', '0.1', '--norm', 'group',
        '--kernel-fwhm-mm', '9', '--resolution-mm', '4', '--cache-dtype', 'float16',
        '--cache-file', CACHE_FILE,
        '--lambda-recon', '1.0', '--lambda-dice', '0.5', '--lambda-topk', '0.5', '--lambda-corr', '0.25',
        '--recon-alpha', '10.0', '--recon-gamma', '1.0',
        '--device', 'cuda' if torch.cuda.is_available() else 'auto',
        '--num-workers', '0',
        '--run-dir', str(run_dir), '--checkpoint-dir', str(Path(run_dir) / 'checkpoints'),
    ]
    if '--use-dilation' in v.get('extra', []):
        args.append('--use-dilation')
    if '--multi-scale' in v.get('extra', []):
        args.append('--multi-scale')
    if '--global-context' in v.get('extra', []):
        idx = v['extra'].index('--global-context')
        args.extend(['--global-context', v['extra'][idx + 1]])
    return args


PLAIN_BEST_VARIANTS = [
    dict(name='ae_pretrained_finetune_cnn_pretrained_text_trainable_plain', model='ale_3dcnn', base_channels=48, num_blocks=4, extra=[]),
]

# Backwards-compatible alias used by older cells. This intentionally contains only the runnable plain best recipe.
VARIANTS = PLAIN_BEST_VARIANTS

GLOBAL_CONTEXT_VARIANTS = [
    dict(name='ae_pretrained_finetune_resnet48_multiscale_attention', model='ale_3dcnn_resnet', base_channels=48, num_blocks=4, extra=['--multi-scale', '--global-context', 'attention']),
    dict(name='ae_pretrained_finetune_resnet64_multiscale_attention', model='ale_3dcnn_resnet', base_channels=64, num_blocks=4, extra=['--multi-scale', '--global-context', 'attention'], ae_batch_size_candidates='128,96,64,32,16,8,4'),
    dict(name='ae_pretrained_finetune_resnet48_dilated_multiscale_attention', model='ale_3dcnn_resnet', base_channels=48, num_blocks=4, extra=['--use-dilation', '--multi-scale', '--global-context', 'attention']),
    dict(name='ae_pretrained_finetune_resnet48_5stage_multiscale_se', model='ale_3dcnn_resnet', base_channels=48, num_blocks=5, extra=['--multi-scale', '--global-context', 'se'], ae_batch_size_candidates='128,96,64,32,16,8,4'),
]


def run_command(cmd, done_file=None, log_file=None):
    if done_file is not None and SKIP_COMPLETED and Path(done_file).exists():
        print('Skipping completed:', done_file)
        return 0
    print(cmd)
    if log_file is None:
        log_file = Path(RUNS_DIR) / f'command_{RUN_STAMP}.log'
    log_file = Path(log_file)
    log_file.parent.mkdir(parents=True, exist_ok=True)
    with log_file.open('w') as f:
        f.write(cmd + '\n\n')
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            f.write(line)
        code = proc.wait()
        f.write(f'\nEXIT_CODE={code}\n')
    if code != 0:
        print(f'Command failed with exit code {code}. Full log: {log_file}')
        try:
            lines = log_file.read_text().splitlines()
            print('--- log tail ---')
            print('\n'.join(lines[-80:]))
            print('--- end log tail ---')
        except Exception as exc:
            print('Could not read log tail:', exc)
    return code


def _latest_existing(paths):
    existing = [Path(p) for p in paths if Path(p).exists()]
    if not existing:
        return None
    return max(existing, key=lambda p: p.stat().st_mtime)


def find_existing_autoencoder_for_variant(v):
    patterns = [
        f'autoencoder_{v["name"]}_*/checkpoints/best_cnn_autoencoder.pt',
        f'autoencoder_{v["name"]}_*/checkpoints/last_cnn_autoencoder.pt',
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(Path(RUNS_DIR).glob(pattern))
    return _latest_existing(candidates)


def find_existing_variant_run(v):
    return _latest_existing(Path(RUNS_DIR).glob(f'{v["name"]}_*'))


def run_variant(v):
    existing_run = find_existing_variant_run(v)
    if existing_run and (existing_run / 'eval_results.json').exists() and SKIP_COMPLETED:
        print('Skipping completed contrastive run:', existing_run)
        return 0
    autoencoder_checkpoint = v.get('autoencoder_checkpoint') or AUTOENCODER_CHECKPOINT or find_autoencoder_checkpoint()
    if not autoencoder_checkpoint:
        print('No existing autoencoder checkpoint found; training a matching autoencoder first:', v['name'])
        autoencoder_checkpoint = pretrain_autoencoder_for_variant(v)
    run_dir = existing_run if existing_run else Path(RUNS_DIR) / f'{v["name"]}_{RUN_STAMP}'
    resume_ckpt = run_dir / 'checkpoints' / 'last_ale_cnn.pt'
    args = base_args(autoencoder_checkpoint) + [
        '--model', v['model'],
        '--base-channels', str(v['base_channels']),
        '--num-blocks', str(v['num_blocks']),
        '--run-dir', str(run_dir),
        '--checkpoint-dir', str(run_dir / 'checkpoints'),
    ] + v.get('extra', [])
    if resume_ckpt.exists() and not (run_dir / 'eval_results.json').exists():
        args.extend(['--resume-from', str(resume_ckpt)])
        print('Resuming contrastive run from:', resume_ckpt)
    cmd = 'python experiments/3dcnn/atlas_free_cnn/training/train_ale_cnn.py ' + ' '.join(map(str, args))
    return run_command(cmd, done_file=run_dir / 'eval_results.json', log_file=run_dir / 'train.log')


def pretrain_autoencoder_for_variant(v):
    existing = v.get('autoencoder_checkpoint')
    if existing and Path(existing).exists():
        print('Using preset autoencoder checkpoint:', existing)
        return existing
    if v.get('model') == 'ale_3dcnn':
        existing = AUTOENCODER_CHECKPOINT or find_autoencoder_checkpoint()
        if existing and Path(existing).exists():
            v['autoencoder_checkpoint'] = existing
            print('Using discovered plain-CNN autoencoder checkpoint:', existing)
            return existing
    existing_variant_ae = find_existing_autoencoder_for_variant(v)
    if existing_variant_ae:
        v['autoencoder_checkpoint'] = str(existing_variant_ae)
        print('Using existing matching autoencoder checkpoint:', existing_variant_ae)
        return str(existing_variant_ae)
    run_dir = Path(RUNS_DIR) / f'autoencoder_{v["name"]}_{RUN_STAMP}'
    best_ckpt = run_dir / 'checkpoints' / 'best_cnn_autoencoder.pt'
    if not best_ckpt.exists():
        args = autoencoder_args(v, run_dir)
        cmd = 'python experiments/3dcnn/atlas_free_cnn/training/train_ale_cnn_autoencoder.py ' + ' '.join(map(str, args))
        code = run_command(cmd, done_file=best_ckpt, log_file=run_dir / 'autoencoder_train.log')
        if code != 0:
            raise RuntimeError(f'Autoencoder pretraining failed: {v["name"]}')
    v['autoencoder_checkpoint'] = str(best_ckpt)
    print('Autoencoder checkpoint ready:', best_ckpt)
    return str(best_ckpt)


def run_global_context_variant(v):
    pretrain_autoencoder_for_variant(v)
    code = run_variant(v)
    if code != 0:
        raise RuntimeError(f'Contrastive variant failed: {v["name"]}')
    return code


In [ ]:
# Stage 1: run the plain best recipe using the original improved plain-CNN autoencoder checkpoint.
for variant in PLAIN_BEST_VARIANTS:
    code = run_variant(variant)
    if code != 0:
        raise RuntimeError(f'Variant failed: {variant["name"]}')


In [ ]:
# Stage 2: for each global-context variant, pretrain the matching autoencoder first,
# then run the contrastive fine-tuning recipe with that matching checkpoint.
for variant in GLOBAL_CONTEXT_VARIANTS:
    run_global_context_variant(variant)
